# Matplotlib 之一：折线图（plot）

---

折线图是**最常用**的图，用来展示**随着某个连续变量变化，另一个量怎么变**——比如温度随日期变化、股价随时间变化。

这一篇先把「能画出一条线」跑通，再把**新手必踩的坑**一个个填平，重点是：**中文为什么显示成方块、怎么让参数听话、pyplot 写法和面向对象写法有什么区别**。

## 学习目标

1. 用 `plt.plot()` 画出一条折线，并理解 `x` / `y` 的对应关系
2. 彻底搞懂**中文显示问题**（这是 matplotlib 头号坑），会写一份可复用的配置
3. 掌握画布 `figure`、标题、坐标轴、图例、网格、刻度等常用参数
4. 能在同一张图上画多条线并加图例
5. 看懂 `plt.xxx()`（pyplot）和 `fig, ax = plt.subplots()`（面向对象）两种写法的区别


---
## 一、最小可用折线图

`plt.plot(x, y)` 接收两组**等长**的数字，`x` 是横坐标、`y` 是纵坐标，相邻两点用线段连起来。

> **原稿问题**：原 notebook 第一格最后是 `plt.show`（少了括号）。`plt.show` 只是"拿到函数本身"，并不会执行它，**图根本不会弹出来**。正确写法是 `plt.show()`。下面我们一律写带括号的版本。


In [ ]:
import matplotlib.pyplot as plt

# 先建一个"画布"（figure），再往上面画线
plt.figure()
# plot 的参数是两组等长列表：x 坐标、y 坐标
plt.plot([1, 2, 3, 4, 5, 6, 7], [17, 17, 18, 19, 20, 18, 15])
plt.show()   # 输出: 一条从 (1,17) 蜿蜒到 (7,15) 的折线


---
## 二、头号坑：中文显示成方块

你运行上面那张图如果标题写中文，大概率看到的是**一堆方块 □□□**。原因很简单：

**matplotlib 的默认字体是 `DejaVu Sans`，它不含中文字形**。遇到中文字符，没字模可画，就只能画方块。

解决思路只有一条：**告诉 matplotlib 用一款"带中文"的字体**。做法是改 `plt.rcParams`（全局配置）：

| 配置项 | 作用 |
|---|---|
| `plt.rcParams['font.sans-serif']` | 指定无衬线字体列表，**按顺序查找第一个装了的** |
| `plt.rcParams['axes.unicode_minus']` | 设为 `False`，否则负号 `-` 也会变成方块 |

不同系统自带的中文字体不一样，所以**给一个列表当兜底**最稳妥：

| 系统 | 常用中文字体 |
|---|---|
| Windows | `SimHei`（黑体）、`Microsoft YaHei`（微软雅黑）、`KaiTi`（楷体）、`FangSong`（仿宋） |
| macOS | `Arial Unicode MS`、`PingFang SC`、`Heiti SC` |
| Linux | `WenQuanYi Zen Hei`（文泉驿）、`Noto Sans CJK SC` |

> **为什么 `axes.unicode_minus = False` 也要加？** 光设中文字体还不够：负号 `-` 在某些中文字体里也缺字模，会显示成方块。把它关掉，负号就走 ASCII 的减号，正常显示。


下面这段是**全项目可复用的"中文配置"**，建议存成代码片段，每个需要中文的 notebook 开头都贴一次：


In [ ]:
import matplotlib.pyplot as plt

# 【关键】一次性配置：支持中文 + 负号
plt.rcParams['font.sans-serif'] = [
    'SimHei',           # 黑体（Windows 常见）
    'Microsoft YaHei',  # 微软雅黑
    'KaiTi',            # 楷体
    'FangSong',         # 仿宋
    'Arial Unicode MS', # macOS 常见
    'DejaVu Sans',      # 跨平台兜底（没有中文，但能保证不崩）
]
plt.rcParams['axes.unicode_minus'] = False  # 负号正常显示

# 测试：现在中文标题/标签都能正常显示了
plt.figure(figsize=(8, 6))
plt.plot([1, 2, 3, 4], [1, 4, 2, 3])
plt.title('这是一个中文标题')   # 输出: 中文正常，不再方块
plt.xlabel('横坐标')
plt.ylabel('纵坐标')
plt.legend(['数据线'])
plt.show()


---
## 三、画布与保存图片

`plt.figure(figsize, dpi)` 创建画布：

- `figsize=(宽, 高)`：单位是**英寸**（不是像素）
- `dpi`：每英寸多少像素，**越大越清晰、文件越大**

想保存成图片用 `plt.savefig('文件名.png')`。**关键顺序：savefig 必须写在 show 之前**，否则保存的是一张空白图（`plt.show()` 会清空当前画布状态）。


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

month = [f"{i}月" for i in range(1, 8)]
temp = [17, 17, 18, 19, 20, 18, 15]

plt.figure(figsize=(20, 8), dpi=80)          # 宽20英寸、高8英寸、分辨率80
plt.plot(month, temp, label="温度")
plt.title("2023年7月前7天温度")
plt.xlabel("月份")
plt.ylabel("温度")
plt.legend()

# 【关键】savefig 放在 show 之前，否则存出来是空白图
plt.savefig("温度折线图.png")
plt.show()   # 输出: 折线图，同时工作目录多出一个 温度折线图.png


---
## 四、折线图常用参数（重点）

`plt.plot()` 和后续的美化函数有一堆参数，下面是把最常用的列成一张表，照表抄就行：

| 函数 / 参数 | 作用 | 常见取值 |
|---|---|---|
| `plot(..., color=)` | 线条颜色 | `'r'`/`'g'`/`'b'`，或 `'purple'`、`'#FF9999'` |
| `plot(..., linestyle=)` | 线型 | `'-'` 实线、`'--'` 虚线、`':'` 点线、`'-.'` |
| `plot(..., marker=)` | 数据点形状 | `'o'` 圆、`'s'` 方、`'^'` 三角、`'*'` 星 |
| `plot(..., linewidth=)` | 线宽 | 数字，如 `2` |
| `plot(..., label=)` | 图例文本 | 字符串，配合 `legend()` 显示 |
| `plt.title(...)` | 标题 | 文字，可加 `fontsize`/`color` |
| `plt.xlabel/ylabel` | 轴标签 | 文字 |
| `plt.grid(True)` | 网格线 | `axis='y'` 只显纵向网格 |
| `plt.xticks/yticks` | 刻度位置+标签 | `rotation` 旋转角度，`fontsize` 字号 |
| `plt.ylim/plt.xlim` | 坐标轴范围 | 如 `plt.ylim(12, 22)` |
| `plt.text(x,y,s)` | 在指定坐标写字 | `ha`/`va` 控制对齐 |


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

month = [f"{i}月" for i in range(1, 8)]
temp = [17, 17, 18, 19, 20, 18, 15]

plt.figure(figsize=(20, 8), dpi=80)
# 【关键】一次把 颜色/线型/数据点/线宽/图例 都设好
plt.plot(month, temp, label="温度", marker="o", linestyle="--",
         linewidth=2, color="purple")
plt.xlabel("月份", fontsize=16)
plt.ylabel("温度", fontsize=16)
plt.grid(True, linestyle="--", alpha=0.3, color="red")   # alpha: 透明度 0~1
plt.legend(loc="upper right", fontsize=16)
plt.title("2023年7月1日至7月7日温度变化折线图", fontsize=24,
          fontweight="bold", color="red")
plt.xticks(range(1, 8), fontsize=16, rotation=45)        # rotation 旋转刻度，避免拥挤
plt.yticks(range(15, 21), fontsize=16)
plt.ylim(12, 22)

# 在每个数据点上标注数值：y+1 让字显示在点的上方
for x, y in zip(month, temp):
    plt.text(x, y + 1, y, fontsize=12, ha="center", va="bottom")

plt.show()   # 输出: 一条带圆点、虚线、网格、数值标注的紫色折线


> **原稿问题**：原稿把 `xticks` 的 `rotation` 设成 `315`，等于把刻度文字倒着转，根本看不清。一般设 `45` 最顺眼。


---
## 五、同一张图画多条线

连续调用两次 `plt.plot()` 就会叠在同一张图上。**每条线都要有 `label`，最后用 `legend()` 一次性把图例调出来**。


In [ ]:
import matplotlib.pyplot as plt
import random

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 模拟 11:00-12:00 每分钟的温度（60 个点）
x = range(60)
y_shanghai = [random.uniform(15, 18) for _ in x]
y_beijing = [random.uniform(5, 8) for _ in x]

plt.figure(figsize=(20, 8), dpi=80)
plt.plot(x, y_shanghai, color="r", linestyle="--", label="上海")
plt.plot(x, y_beijing, label="北京")
x_label = [f"11:{i}" for i in x]
plt.xticks(x[::5], x_label[::5])     # 每 5 个刻度标一次，否则太密
plt.yticks(range(0, 35, 5))
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower left")
plt.xlabel("Time")
plt.ylabel("Temperature")            # 输出: 红虚线(上海)+蓝实线(北京)两条曲线
plt.title("上海与北京11:00-12:00每分钟温度变化")
plt.show()


> **原稿问题**：原稿英文标签写成 `Tempture`（拼写错误）。虽然不影响运行，但作为教学材料应修正为 `Temperature`。


---
## 六、两种写法：pyplot  vs  面向对象（重点）

前面全程用 `plt.xxx()`，这叫 **pyplot 写法**（状态机风格，matplotlib 内部帮你记"当前画布/当前轴"）。还有一种是 **面向对象写法**：你自己拿到 `fig` 和 `axes` 对象，方法调用写在对象上。

| 对比 | pyplot 写法 | 面向对象写法 |
|---|---|---|
| 拿画布/坐标轴 | `plt.figure()` 后靠全局状态 | `fig, ax = plt.subplots()` 显式拿到 |
| 画线 | `plt.plot(...)` | `ax.plot(...)` |
| 设标题 | `plt.title(...)` | `ax.set_title(...)` |
| 设刻度 | `plt.xticks(...)` | `ax.set_xticks(...)` + `ax.set_xticklabels(...)` |
| 适合 | 快速画一张图 | **多子图、复杂排版**（见"生成多张图"那篇） |

一句话：**画一张图用 `plt` 够快；画多张/要精细控制，用 `fig, ax` 更清楚**。下面用面向对象重写上面的双城对比，并排成两张子图：


In [ ]:
import matplotlib.pyplot as plt
import random

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

x = range(60)
y_shanghai = [random.uniform(15, 18) for _ in x]
y_beijing = [random.uniform(5, 8) for _ in x]
x_label = [f"11:{i}" for i in x]

# 【关键】一次性拿到"图(fig)"和"两个坐标轴(axes 数组)"
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 8), dpi=80)

axes[0].plot(x, y_shanghai, color="r", linestyle="--", label="上海")
axes[1].plot(x, y_beijing, label="北京")

# 面向对象：用 set_ 系列方法
axes[0].set_xticks(x[::5]); axes[0].set_xticklabels(x_label[::5])
axes[1].set_xticks(x[::5]); axes[1].set_xticklabels(x_label[::5])
axes[0].set_yticks(range(0, 35, 5)); axes[1].set_yticks(range(0, 35, 5))
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(loc="lower left"); axes[1].legend(loc="lower left")
axes[0].set_xlabel("Time"); axes[0].set_ylabel("Temperature")
axes[1].set_xlabel("Time"); axes[1].set_ylabel("Temperature")
axes[0].set_title("上海温度")
axes[1].set_title("北京温度")
plt.show()   # 输出: 左右并排两张子图


---
## 七、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 中文成方块 | 标题/标签是 □ | 设 `font.sans-serif` 中文字体列表 |
| 负号成方块 | `-5` 显示成 □ | 加 `axes.unicode_minus = False` |
| `plt.show` 不显示 | 图没弹出来 | 少了括号，写成 `plt.show()` |
| `savefig` 存空白 | 图片是全白 | savefig 写在 show **之前** |
| 多条线没图例 | 只有线没有说明 | 每条 `plot(..., label=)` + 最后 `legend()` |
| 刻度挤成一团 | 标签重叠看不清 | `plt.xticks(..., rotation=45)` |
| 两条线共用一个 label | 图例错乱 | 每条线单独给 `label` |


---
## 八、动手练习

**练习 1**：用 `plt.plot` 画出 `y = x**2`（`x` 取 0~10 的整数），线型用虚线、数据点用圆圈。

**练习 2**：给练习 1 的图加上中文标题"平方函数"、x 轴"x"、y 轴"y"，并打开网格。

**练习 3**：在同一张图里画 `sin` 和 `cos`（提示：`import numpy as np`，`x = np.linspace(0, 2*np.pi, 100)`），并加图例区分。

**练习 4**：把练习 3 用面向对象写法 `fig, ax = plt.subplots()` 重写一遍。

**练习 5**：用 numpy 生成 30 个 0~20 之间的随机整数当"销量"，画出折线图，并在每个点上方标注数值。

---

**答案思路（先自己做）**

```python
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 1+2
x = list(range(11))
plt.figure()
plt.plot(x, [i**2 for i in x], linestyle="--", marker="o", label="x^2")
plt.title("平方函数"); plt.xlabel("x"); plt.ylabel("y")
plt.grid(True); plt.legend(); plt.show()

# 3
t = np.linspace(0, 2*np.pi, 100)
plt.figure()
plt.plot(t, np.sin(t), label="sin")
plt.plot(t, np.cos(t), label="cos")
plt.legend(); plt.show()

# 5
sales = np.random.randint(0, 21, 30)
plt.figure()
plt.plot(sales, marker="o")
for i, v in enumerate(sales):
    plt.text(i, v + 0.3, v, ha="center")
plt.show()
```
